# TrOCR run 2: aligned token loss, attention LoRA, best CER
Pinned code, model and data; failure stops the notebook. Save outputs before session expiry.

Select GPU T4. CUDA computation is checked before downloading data/models. The custom trainer shifts decoder inputs once and compares logits with labels at the same token positions.

In [ ]:
import subprocess, sys, os
from pathlib import Path
CODE_REVISION = "90e1a9643a9443f0dc113ad94ce9a5d5456e52de"
BASE_REVISION = "93450be3f1ed40a930690d951ef3932687cc1892"
DATA_REVISION = "d881debb90045fd71ad8e25faeeafeb6adab6622"
repo = Path('/kaggle/working/OCR_engine')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/PiotrStyla/OCR_engine.git',str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','origin',CODE_REVISION],check=True)
subprocess.run(['git','-C',str(repo),'checkout','--detach',CODE_REVISION],check=True)
os.chdir(repo)
sys.path.insert(0,str(repo))
subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=True)
subprocess.run([sys.executable,'-m','pip','install','transformers==4.57.6','peft==0.19.1','jiwer','pillow','accelerate'],check=True)

In [ ]:
import tarfile
import torch
assert torch.cuda.is_available(), "GPU is required; select Kaggle GPU T4."
print('GPU:', torch.cuda.get_device_name(0), 'torch:', torch.__version__, 'architectures:', torch.cuda.get_arch_list())
try:
    probe = torch.ones((2, 2), device='cuda')
    assert (probe @ probe).sum().item() == 8.0
    torch.cuda.synchronize()
    del probe
except Exception as exc:
    raise RuntimeError('GPU cannot execute this PyTorch build. Select Kaggle T4; P100 sm_60 is unsupported.') from exc
print('CUDA_PREFLIGHT_OK', flush=True)
from huggingface_hub import hf_hub_download
from training.protocol import pair_manifest
archive = hf_hub_download(
    'PiotrSty/ocr-pl-lines',
    'ocr-pl-lines-v1.tar.gz',
    repo_type='dataset',
    revision=DATA_REVISION,
)
data_root = Path('/kaggle/working/ocr-pl-lines-v1')
data_root.mkdir(parents=True, exist_ok=True)
with tarfile.open(archive, 'r:gz') as bundle:
    bundle.extractall(data_root, filter='data')
print('train:', len(pair_manifest(data_root/'train')), 'val:', len(pair_manifest(data_root/'val')))

In [ ]:
# Run after inspecting first-run reevaluation. Start fresh from the pinned base.
output = '/kaggle/working/trocr-pl-run2'
subprocess.run([sys.executable,'-m','training.train_trocr_pl',
    '--train-dir',f'{data_root}/train','--val-dir',f'{data_root}/val',
    '--revision',BASE_REVISION,'--output',output,
    '--epochs','3','--batch-size','8','--no-4bit'],check=True)
print(Path(output,'selection.json').read_text())
print(Path(output,'best_metrics.json').read_text())
# No upload_folder: preserve the first model; review CER before any promotion.
